# Equity & cross-asset risk premia — ERP 0.13

**Research sequence:** standard FF/AQR benchmark; residual dependence and pricing-oriented factor discovery; macro proxy and identified policy shock analysis; rolling portfolio construction; fixed/rolling/expanding FMB, GMM and three-pass pricing; Gaussian shrinkage; funded Rates and traded cross-asset proxy integration.

Uses the checked-out ERP 0.13.0 source. All model parameters below are fixed for this research run; historical evaluation is retrospective, not independent confirmation. Characteristic-sorted portfolio groups may overlap at the stock level. RP-PCA and Giglio–Xiu are adaptations, not full published inference. FMB lacks Shanken correction; regularized GMM does not support a conventional J-test. FRED changes are proxy shocks, not CPI release surprises; data are revised histories, not publication vintages. Factor and asset returns are monthly decimals unless explicitly annualized.

See the root README for setup and `results/evidence/` for the executed historical run.

In [ ]:
# Load checked-out ERP source. Install the repository first; see README.
import sys, subprocess, importlib.util
from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd
roots = [Path.cwd(), *Path.cwd().parents, Path('/content/empirical-risk-premia-engine')]
REPO = next((p for p in roots if (p/'src'/'erp'/'__init__.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Open from the cloned repository or extract the release to /content/empirical-risk-premia-engine; see README.')
sys.path.insert(0, str(REPO/'src'))
for key in list(sys.modules):
    if key == 'erp' or key.startswith('erp.'):
        del sys.modules[key]
from erp import (__version__,EquityData,analyze_equity_residuals,
    analyze_equity_factor_extensions,load_equity_fred_macro,
    analyze_equity_v012,load_jk_monetary_shocks,
    load_inflation_release_surprises,analyze_equity_shock_mapping,
    estimate_jk_shock_prices)
print('ERP source:', __version__)

from erp import analyze_rolling_shocks
from erp.equity_macro import find_complete_equity_fred_file
assert __version__ == "0.13.0"

from erp import (analyze_dynamic_premia,constrained_rp_rolling,gx_initial_inference,
    economic_return_decomposition,load_etf_adjusted_prices,etf_factor_returns,
    load_rates_factor_returns,managed_factor_returns,RatesData,RatesPremiaEngine)
from erp.equity_extensions import _aligned
from erp.equity_residual import _hac_alpha,_holm

assert __version__ == "0.13.0"


## 1. Fixed discovery / test design

First 80 portfolios: 25 Size–B/M, 30 Industries, 25 Size–Momentum. Holdout 50: 25 Size–Profitability + 25 Size–Investment. **Constituent stocks can overlap** despite distinct portfolio definitions. Train through December 2005, OOS from January 2006. `GAMMAS` and `N_PCS` **must not be selected by validation performance**.


In [ ]:
MODEL='FF6_BAB_QMJ'
AS_OF='2026-10-31'
TRAIN_END='2005-12-31'
DISCOVERY=('SIZE_BM_25','INDUSTRY_30','SIZE_MOM_25')
VALIDATION=('SIZE_OP_25','SIZE_INV_25')
N_PCS=5
GAMMAS=(0.,12.,60.) # predeclared gamma=0 is the original PCA baseline
NULL_DRAWS=199
NULL_SEED=7331
MACRO_STATE_LAG_MONTHS=2
AQR_PATHS={} # original AQR workbook fallback if source blocked
FRED_LOCAL_PATH=None # OPTIONAL: original combined FRED CSV/ZIP from browser
CACHE=Path('erp_cache')
OUT=Path('erp_equity_v013_results');OUT.mkdir(parents=True,exist_ok=True)

SHOCK_TRAIN_END="2011-12-31"
PROMPT_FRED_UPLOAD=True # set False for noninteractive runs

# New experiments: frozen hyperparameters, no OOS tuning.
DYNAMIC_WINDOW=72
DYNAMIC_MIN_TRAIN=60
DYNAMIC_REFIT=3
DYNAMIC_GX_K=8
DYNAMIC_BOOT_DRAWS=199
RUN_CROSS_ASSET_ETFS=True
RUN_FUNDED_RATES=True
ETF_ADJUSTED_CSV=None # optional local CSV with Date,IEF,HYG,UUP,DBC,VIXY adjusted USD closes
RATES_FACTORS_CSV=None # optional exact ERP Rates factor_returns.csv
RATES_CURVE_PATH=None # optional original Kim-Wright source
RATES_FUNDING_PATH=None # optional original FEDFUNDS CSV


## 2. Published Equity data, baseline v0.10 (unchanged)

The baseline remains untouched. Factor portfolios are fixed after formation, not continuously traded at zero cost. Neutrality against model factors holds in training, not necessarily OOS.

In [ ]:
data=EquityData.from_public_sources(cache_dir=str(CACHE),aqr=True,aqr_paths=AQR_PATHS)
base=analyze_equity_residuals(data,model=MODEL,discovery_universes=DISCOVERY,
    validation_universes=VALIDATION,train_end=TRAIN_END,as_of=AS_OF,n_pcs=N_PCS)
print('Training/OOS months:',base.metadata['formation_months'],base.metadata['oos_months'])
print('Original residual PC premia:');display(base.fmp_premia_tests.round(5))
print('Baseline mean-pricing errors / attribution:');display(base.holdout_metrics.round(5))


## 3. Test the existence of synchronous residual structure

Circular-shift each discovery-asset residual independently `NULL_DRAWS` times, preserving each asset's time path and autocorrelation while destroying synchronous cross-asset dependence. The rankwise eigenvalue null and conservative max-eigenvalue FWER comparison test common residual *covariance*, **not priced risks**. Compare principal subspaces between two formation-only halves to assess instability; this is not a formal confidence interval.

In [ ]:
ext=analyze_equity_factor_extensions(data,base,n_pcs=N_PCS,null_draws=NULL_DRAWS,
    null_seed=NULL_SEED,rp_gamma=GAMMAS,gx_k=N_PCS)
print('Residual eigenvalue null:');display(ext.eigen_null.round(6))
print('Formation subspace stability:');display(ext.subspace_stability.round(5))


## 4. Pricing-oriented residual factor challengers + SDF + three-pass

Formation covariance `C` is augmented by `gamma × a a′` using **training OLS alpha vector** `a` rather than zero-mean residual scores. This is an RP-PCA-*inspired* test: not the precise Lettau–Pelger procedure. Each direction is projected into net-dollar and FF/AQR-beta neutral portfolios using training betas. Mean and spanning-alpha tests use OOS months and HAC; Holm correction covers the whole predeclared gamma × PC family. Held-out asset means and SDF/GMM are also scored OOS.

Giglio–Xiu-style three passes: return PCA, cross-sectional lambda from training heldout assets, projection of **observed factors** onto latent factors. It is a point-estimate research diagnostic, not weak-factor robust inference.

In [ ]:
print('Predeclared pricing-oriented portfolios:');display(ext.pricing_oriented_premia.round(5))
print('Heldout mean-pricing errors, regularized SDF:');display(ext.pricing_oriented_sdf.round(5))
print('Three-pass observed FF/AQR premia diagnostics:');display(ext.three_pass_observed.round(5))
print('Three-pass validation errors:');display(ext.three_pass_diagnostics.round(5))


## 5. Download macro & cross-asset proxies automatically from FRED

`CPIAUCSL`: Consumer Price Index; `INDPRO`: Industrial Production; `DGS10` and `DGS2`: Treasury yields; `BAA10Y`: Moody's Baa–10Y Treasury spread; `VIXCLS`: CBOE volatility index. Monthly official observations are resampled without forward fill. Features: log YoY CPI/IP, 10Y–2Y slope, credit spread and log VIX. **First differences are statistical economic changes, not event-study identified macro surprises.**

The loader attempts a combined FRED download, then per-series downloads, then caches. If Colab cannot reach FRED, download the **original** combined CSV from `https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL%2CINDPRO%2CDGS10%2CDGS2%2CBAA10Y%2CVIXCLS`, place its path in `FRED_LOCAL_PATH`, and re-run this cell. Do not fill unavailable values or substitute synthetic data.
**Colab file uploads:** if you have uploaded `fredgraph (5).zip` (or any `fredgraph*.zip` / `fredgraph*.csv`) in the `/content` Files panel, the next cell automatically discovers it and bypasses FRED downloads. You may still set `FRED_LOCAL_PATH` explicitly. Upload the file before executing this cell.


In [ ]:
# Select a complete Equity file, never accidentally a partial Rates download.
from pathlib import Path
macro=None
try:
    fred_path,rejected=find_complete_equity_fred_file([Path('/content'),Path.cwd(),CACHE/'macro'],explicit=FRED_LOCAL_PATH)
    print('Rejected local candidates:',rejected)
    if fred_path is not None: print('Using local FRED:',fred_path)
    try:
        macro=load_equity_fred_macro(cache_dir=CACHE/'macro',path=fred_path,as_of=AS_OF,timeout=8)
    except (ConnectionError,ValueError,FileNotFoundError) as exc:
        if not PROMPT_FRED_UPLOAD: raise
        try: from google.colab import files
        except ImportError: raise exc
        print('Upload the original combined six-series FRED CSV/ZIP (e.g. fredgraph (5).zip). Cancel to continue without FRED.')
        print('https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL,INDPRO,DGS10,DGS2,BAA10Y,VIXCLS')
        uploaded=files.upload()
        for filename in uploaded:
            try:
                path,_=find_complete_equity_fred_file([],explicit=filename)
                macro=load_equity_fred_macro(cache_dir=CACHE/'macro',path=path,as_of=AS_OF)
                break
            except (ValueError,OSError) as e: print('Rejected',filename,e)
    if macro is not None:
        print('FRED source:',macro.provenance['source']); display(macro.states.tail(6));display(macro.changes.tail(6))
    else: print('FRED not estimated; no data substituted.')
except (ConnectionError,FileNotFoundError,ValueError) as exc:
    print('FRED not estimated:',exc)


## 6. FF and RPC economic factor mapping + conditional premia

**Contemporaneous interpretation:** regress FF/AQR and RPC returns on *monthly changes* of macro/cross-asset variables using formation dates, then evaluate the estimated mapping on OOS months. A high R² does **not** identify a causal macro risk premium.

**Predictability:** state variables are lagged by two complete calendar months and a Ridge forecast of factor expected returns is refitted expanding-window. Its OOS R² is compared with an expanding unconditional mean. Data are latest revised FRED historical observations, **not point-in-time vintages**. The optional one-step SDF numbers using economic changes are **proxy-priced moment diagnostics**, not validated prices of economically identified macro risks.

In [ ]:
macro_result=None
if macro is not None:
    macro_result=analyze_equity_residuals(data,model=MODEL,discovery_universes=DISCOVERY,
        validation_universes=VALIDATION,train_end=TRAIN_END,as_of=AS_OF,
        n_pcs=N_PCS,macro_shocks=macro.changes,state_predictors=macro.states,
        state_lag_months=MACRO_STATE_LAG_MONTHS)
    print('FF + RPC factor exposures to economic changes (NOT risk prices):')
    display(macro_result.macro_exposures.round(5))
    print('Cross-time stability / interpretation diagnostics:')
    display(macro_result.macro_diagnostics.round(5))
    print('Lagged macro-state factor return forecasting, OOS vs unconditional:')
    display(macro_result.state_scores.round(5))
    print('Tentative one-step macro proxy SDF — NOT identified shock risk prices:')
    display(macro_result.macro_sdf_pricing.round(5))
else:
    print('Macro mapping and forecast remain UNESTIMATED; no proxy or synthetic input used.')


## 7. v0.11 baseline exports and interpretation

All CSVs are separate. Conclude only when residual eigenvalue significance, OOS pricing, and OOS conditional prediction each survive their own tests. Neither residual R² nor FF6 alpha alone proves a new priced factor. No TAA factor rotation alpha is claimed.

In [ ]:
base.export(OUT/'equity_baseline_v010')
ext.export(OUT/'equity_extensions_v011')
if macro is not None:
    macro.export(OUT/'public_FRED_inputs')
    (OUT/'public_FRED_inputs'/'source_metadata.json').write_text(__import__('json').dumps(macro.provenance,indent=2))
    if macro_result is not None:macro_result.export(OUT/'equity_economic_mapping')
print('Saved to:',OUT.resolve())
for x in sorted(OUT.rglob('*.csv')):print(' -',x)


## 8. Rolling vs static: are the residual factors stable, and is pricing orientation useful?

**Freeze only at each formation date:** rolling windows of 120 months, rebalance annually. Each FMP is dollar-neutral and neutral to formation FF/AQR betas, using actual discovery-portfolio excess returns. Signs are aligned to the *previously known* factor direction. Annualized OOS premia are **not** automatically priced systematic risks. Prespecified gamma=0 vs gamma=12, first 3 PCs; no tuning on OOS results.

In [ ]:
ROLLING_WINDOW_MONTHS=120
ROLLING_REBALANCE_MONTHS=12
V012_N_PCS=3
V012_GAMMAS=(0.,12.)
pricing=analyze_equity_v012(data,base,formation_window=ROLLING_WINDOW_MONTHS,
    rebalance_months=ROLLING_REBALANCE_MONTHS,n_pcs=V012_N_PCS,
    gammas=V012_GAMMAS,bootstrap_draws=399)
print('Rolling PC premia:');display(pricing.rolling_premia.round(5))
print('First/last formation, turnover and FF neutrality:')
display(pricing.rolling_diagnostics.head(6).round(5))
display(pricing.rolling_diagnostics.tail(6).round(5))
pricing.export(OUT/'equity_v012_rolling_and_pricing')


## 9. FMB vs two-step GMM on precisely the same holdout assets and dates

Compare FF6/BAB/QMJ alone, FF+3 *plain residual PCs* (gamma=0), and FF+3 *augmented-mean PCs* (gamma=12). FMB formation betas and mean lambda estimates are fixed before the OOS period; two-step GMM uses shrinkage HAC weighting and the SDF pricing moments. Rolling FMP returns are **not** retrospectively inserted into the fixed-formation horse race. The bootstrap resamples **monthly loss differences** in common 12-month blocks; do not select gamma based on OOS results.

In [ ]:
print('Common-sample pricing: annualized mean RMSE, FMB vs two-step GMM')
display(pricing.pricing_comparison.round(6))
print('FMB training lambda estimates and HAC t-stats:')
display(pricing.fmb_lambdas.round(5))
print('Paired block-bootstrap differences vs original FF model:')
display(pricing.bootstrap_comparison.round(7))


## 10. Actual FOMC policy-information shocks — Jarociński–Karadi

The updated [author dataset](https://github.com/marekjarocinski/jkshocks_update_fed) contains monthly *MP_median* and *CBI_median* high-frequency announcement shocks. The two add to the announcement policy surprise. The panel is monthly since 1990, **not the same as changes in the level of the policy rate**; source currently ends March 2026, so months after that are missing, not zero. First attempt public GitHub, then cached copy, then a local original CSV (`shocks_fed_jk_m.csv`). This is independent of FRED macro download.

**Same-month fit** measures shock-factor comovement and is not a tradeable month-ahead predictor. The lag-1 experiment tests whether shocks observed during month t predict factor returns in month t+1. These are observational factor-response diagnostics, not yet an identified macro risk-premium estimator.
**Hotfix:** Original-source JK months with failed decomposition are excluded as missing (not zero) and recorded in shock provenance.


In [ ]:
from pathlib import Path
# Optional, if GitHub unavailable: upload author CSV and use its actual path here.
JK_LOCAL_PATH=None
if JK_LOCAL_PATH is None:
    candidates=sorted(Path('/content').glob('shocks_fed_jk_m*.csv')) if Path('/content').exists() else []
    if candidates:
        JK_LOCAL_PATH=str(candidates[-1]);print('Found uploaded JK file:',JK_LOCAL_PATH)
jk=None;shock_mapping=None
try:
    jk=load_jk_monetary_shocks(cache_dir=CACHE/'shock_data',path=JK_LOCAL_PATH,as_of=AS_OF)
    print('JK source:',jk.provenance['source'],'ends',jk.shocks.index.max())
    print('Source months excluded (unknown shocks, not zero):',jk.provenance.get('excluded_invalid_decomposition_months',[]))
    f,_=data.factors(MODEL)
    shock_mapping=analyze_equity_shock_mapping(f,pd.DataFrame(index=f.index),jk,
        train_end=TRAIN_END)
    if shock_mapping.empty:
        print('Insufficient common train/OOS months for JK analysis')
    else:
        display(shock_mapping.round(5))
    jk.export(OUT/'jk_announcement_shocks')
    shock_mapping.to_csv(OUT/'jk_legacy_ff_exposure_and_predictability.csv',index=False)
    # Pricing: nontraded-shock beta prices inferred cross-sectionally, FF controls.
    # This is a point-estimate diagnostic under high-frequency identifying assumptions.
    _,rf=data.factors(MODEL)
    holdout=pd.concat([data.excess_returns(k,rf).add_prefix(k+'__')
         for k in VALIDATION],axis=1)
    shock_price_summary,shock_price_lambda=estimate_jk_shock_prices(
         holdout,f,jk,train_end=TRAIN_END)
    print('JK nontraded-shock prices, heldout common sample (exploratory):')
    display(shock_price_summary.round(6))
    display(shock_price_lambda.loc[shock_price_lambda.factor.isin(
         ['MP_median','CBI_median'])].round(6))
    shock_price_summary.to_csv(OUT/'jk_common_sample_mean_pricing.csv',index=False)
    shock_price_lambda.to_csv(OUT/'jk_shock_fmb_lambda_hac.csv',index=False)
except (ConnectionError,ValueError,FileNotFoundError) as e:
    print('JK shocks unavailable: ',e)
    print('Download https://raw.githubusercontent.com/marekjarocinski/jkshocks_update_fed/main/shocks_fed_jk_m.csv')
    print('Upload the original CSV and re-run this cell; FRED levels are not used as a substitute.')


## 11. Inflation *release* surprises — optional separate input

There is no reliable automatic public consensus archive in this release. **Do not substitute** FRED CPI changes or Cleveland Fed expectations for verified announcement surprises. If you possess release-level data with `release_date,actual,consensus,consensus_source` (same rate units, pre-release consensus), load the CSV/ZIP. Results are separate from monetary-policy shocks. The parser cannot independently verify when a provider recorded its consensus.

In [ ]:
CPI_RELEASE_CSV_PATH=None
inflation_shocks=None
if CPI_RELEASE_CSV_PATH is not None:
    inflation_shocks=load_inflation_release_surprises(CPI_RELEASE_CSV_PATH,as_of=AS_OF)
    ff,_=data.factors(MODEL)
    cpi_exposures=analyze_equity_shock_mapping(ff,pricing.rolling_returns,
        inflation_shocks,train_end=TRAIN_END,min_oos=24)
    display(cpi_exposures.round(5))
    inflation_shocks.export(OUT/'inflation_release_surprises')
else:
    print('CPI announcement surprises NOT estimated: no verified actual/consensus release file')


## 12. Dedicated rolling-factor economic test — new in v0.12.2

All six rolling factors are now genuinely observed during the dedicated 2006–2011 mapping training period. Use a common complete-case calendar for FF, RP, holdout assets and current/lagged shocks. Pricing compares FF, FF+JK, FF+each RP family and FF+each RP family+JK under FMB and GMM. The shock-only SDF contribution is a descriptive covariance-pricing projection calibrated on the holdout portfolios, not structural causal decomposition. Historical OOS dates are reused from earlier research. HAC uses consecutive complete-case observations; weak-identification inference is not validated.

In [ ]:
ff,rf=data.factors(MODEL)
holdout=pd.concat([data.excess_returns(k,rf).add_prefix(k+'__') for k in VALIDATION],axis=1)
rolling_economic={}
economic_inputs={}
if jk is not None: economic_inputs['JK']=jk.shocks[['MP_median','CBI_median']]
if macro is not None: economic_inputs['FRED_PROXY']=macro.changes
for label,shock_panel in economic_inputs.items():
    try:
        result=analyze_rolling_shocks(ff,pricing.rolling_returns,holdout,shock_panel,
            train_end=SHOCK_TRAIN_END,as_of=AS_OF,group_label=label)
        rolling_economic[label]=result
        result.export(OUT/('rolling_economic_'+label))
        print(label,'common sample:',result.metadata)
        print('Factor mapping (all FF and RP, including FF-controlled RP):');display(result.mapping.round(5))
        print('Heldout pricing on identical dates:');display(result.pricing.round(6))
        print('FMB risk prices (exploratory):');display(result.lambdas[result.lambdas.factor.isin(shock_panel.columns)].round(6))
        print('Shock-only SDF implied premia, not causal attribution:');display(result.premium_mapping.round(5))
    except ValueError as e: print(label,'dedicated RP economic study not estimated:',e)
if not rolling_economic: print('No dedicated economic test completed; inspect input/coverage messages.')


## 13. Constrained RP-PCA: objective and stability

Use the balanced-panel Lettau–Pelger moment matrix $R^{\top}R/T+\gamma\bar r\bar r^{\top}$ (gamma=-1 is centered PCA). Here the asset-weight space is restricted **before** extraction to zero-net-dollar and FF-neutral directions. This is a constrained adaptation, not a claim of complete published asymptotic inference. Gamma -1 and 11 match the legacy mean weights 0 and 12; normalization and assignment differ. Re-estimate on trailing 120 months annually; match columns by maximum absolute overlap to prior formation. Preserve all old RP returns. Report HAC alpha and Holm across the six new candidates (retrospective).

In [ ]:
ff_all,discovery_all,holdout_all,_=_aligned(data,base)
crp,crp_diag,crp_w=constrained_rp_rolling(data,base,window=120,step=12,k=3,gammas=(-1.,11.))
crp.to_csv(OUT/'constrained_rp_returns.csv');crp_diag.to_csv(OUT/'constrained_rp_diagnostics.csv',index=False);crp_w.to_csv(OUT/'constrained_rp_weights.csv')
rows=[]
for col in crp:
    a=_hac_alpha(crp[col],ff_all);m=_hac_alpha(crp[col]);rows.append({'factor':col,'mean_ann':12*m['alpha_monthly'],'alpha_ann':12*a['alpha_monthly'],'t_hac':a['t_hac'],'p_hac':a['p_hac'],'nobs':a['nobs']})
crp_premia=pd.DataFrame(rows).set_index('factor');crp_premia['holm_p_new_candidates']=_holm(crp_premia.p_hac)
crp_premia.to_csv(OUT/'constrained_rp_premia.csv');display(crp_premia.round(5));display(crp_diag.tail(6))
rp_all=pd.concat([pricing.rolling_returns,crp],axis=1)


## 14. Three-pass prices (Giglio–Xiu-style) and uncertainty

Centered PCA of discovery returns; no-intercept cross-sectional pricing of latent betas on formation holdout returns; projection of observed factors on latent innovations. Three fixed latent dimensions, synchronized blocks with all three stages refit. Projection R² and condition number diagnose identification, not a formal weak-factor-robust test. Bootstrap intervals are exploratory, not published asymptotic inference.

In [ ]:
gx_initial=gx_initial_inference(ff_all,rp_all,holdout_all,discovery_all,k_values=(5,8,12),train_end=SHOCK_TRAIN_END,as_of=AS_OF,draws=DYNAMIC_BOOT_DRAWS)
gx_initial.to_csv(OUT/'gx_initial_block_inference.csv',index=False);display(gx_initial.round(5))


## 15. Cross-asset traded-return inputs

Use original funded ERP Rates factors (Kim–Wright/FEDFUNDS) and a **separate** ETF proxy panel: IEF excess, HYG minus IEF (not duration-neutral), UUP excess (USD futures fund), DBC excess (commodity futures fund), VIXY excess (volatility futures fund, not a variance swap). All ETF inputs are adjusted USD close prices and all cash-funded legs subtract the French RF. No pre-inception backfill or substitute data. Source failures are logged; you can supply exact local CSVs. For revised macro states, add a separate managed-return experiment MKT/HML times CPI inflation and term slope with two-month lag and trailing standardization. No CPI release surprises are manufactured.

In [ ]:
extra_panels={};extra_sources=[]
if jk is not None:extra_panels['JK']=(jk.shocks[['MP_median','CBI_median']],False)
if macro is not None:
    extra_panels['FRED_PROXY']=(macro.changes,False)
    extra_panels['MANAGED']=(managed_factor_returns(ff_all,macro.states),True)
if RUN_FUNDED_RATES:
    try:
        if RATES_FACTORS_CSV is not None:
            rate_factors=load_rates_factor_returns(RATES_FACTORS_CSV);rate_source=str(RATES_FACTORS_CSV)
        else:
            rd=RatesData.from_fred(cache_dir=CACHE/'rates',local_curve_path=RATES_CURVE_PATH,local_fedfunds_path=RATES_FUNDING_PATH)
            engine=RatesPremiaEngine(rd);engine.fit(as_of=AS_OF);rate_factors=engine.factor_returns.add_prefix('XA_RATES_');rate_source='ERP funded Kim-Wright / FEDFUNDS'
        extra_panels['FUNDED_RATES']=(rate_factors,True);extra_sources.append({'panel':'FUNDED_RATES','source':rate_source,'status':'available'})
    except (ConnectionError,OSError,ValueError,RuntimeError) as e:
        extra_sources.append({'panel':'FUNDED_RATES','status':'unavailable','detail':str(e)});print('Rates unavailable:',e,'; supply original ERP factor_returns.csv or source data.')
if RUN_CROSS_ASSET_ETFS:
    try:
        if ETF_ADJUSTED_CSV is None:
            candidate=Path('/content/etf_adjusted_prices.csv')
            if candidate.is_file(): ETF_ADJUSTED_CSV=str(candidate)
        if ETF_ADJUSTED_CSV is None:
            import importlib.util,subprocess,sys
            if importlib.util.find_spec('yfinance') is None:
                subprocess.run([sys.executable,'-m','pip','-q','install','yfinance'],check=True,timeout=120)
        px,pxsource=load_etf_adjusted_prices(cache_dir=CACHE/'cross_asset',path=ETF_ADJUSTED_CSV,as_of=AS_OF)
        xa=etf_factor_returns(px,rf,as_of=AS_OF);extra_panels['ETF_PROXY']=(xa,True);extra_sources.append({'panel':'ETF_PROXY','source':pxsource,'status':'available'})
    except Exception as e:
        extra_sources.append({'panel':'ETF_PROXY','status':'unavailable','detail':str(e)});print('ETF proxy data unavailable:',e,'; supply etf_adjusted_prices.csv with Date, IEF,HYG,UUP,DBC,VIXY adjusted closes.')
# A combined rates + ETFs experiment is distinct from the two standalone panels.
if 'FUNDED_RATES' in extra_panels and 'ETF_PROXY' in extra_panels:
    extra_panels['JOINT_XA']=(pd.concat([extra_panels['FUNDED_RATES'][0],extra_panels['ETF_PROXY'][0].drop(columns='XA_DURATION')],axis=1),True)
for name,(panel,traded) in extra_panels.items():
    panel.loc[:AS_OF].to_csv(OUT/(name+'_raw_factors.csv'));print(name,'complete months:',len(panel.loc[:AS_OF].dropna()),'traded-return proxy:',traded)
pd.DataFrame(extra_sources).to_csv(OUT/'cross_asset_sources.csv',index=False)


## 16. Dynamic prices and posterior shrinkage — same forecast dates within each panel

Each quarterly observed origin uses historical data only. Fixed, expanding, and trailing 72-calendar-month estimates use the same forecast dates and holdout assets. Regressions estimate formation betas and FMB prices; GMM is two-step HAC-weighted shrinkage. FMB_SHRINK applies a zero-centered Gaussian prior with SD 0.25 times monthly factor volatility (intercept weakly regularized) to estimated prices using their HAC covariance. DIRECT_MEAN and DIRECT_SHRINK forecast traded FMP returns directly; no direct-mean target for nontraded shocks. GX_LATENT is a separate latent pricing model, not an FF regression. Extras are neutralized to FF on formation, preserving intercept; transformed return targets depend on hedge scheme. Mean-pricing RMSE, monthly forecast RMSE and factor forecasts are separate; bootstrap compares paired OOS mean errors conditional on fitted forecasts. Cross-panel scores are **not** directly comparable because start dates may differ. No trading strategy or costs yet.

In [ ]:
dynamic_summaries=[]; dynamic_boots=[]; dynamic_factor_scores=[]; status=[]
# Full equity comparison including the two constrained challengers. Economic panels keep legacy RP0/RP12
# to isolate the effect of pricing dynamics and the new economic drivers.
experiments={'EQUITY':(None,False),**extra_panels}
for name,(panel,traded) in experiments.items():
    try:
        rp_input=rp_all if name=='EQUITY' else pricing.rolling_returns
        result=analyze_dynamic_premia(ff_all,rp_input,holdout_all,discovery_all,extras=panel,extra_traded=traded,
            as_of=AS_OF,window=DYNAMIC_WINDOW,min_train=DYNAMIC_MIN_TRAIN,step=DYNAMIC_REFIT,k=DYNAMIC_GX_K,draws=DYNAMIC_BOOT_DRAWS)
        folder=OUT/('dynamic_'+name);result.export(folder)
        dynamic_summaries.append(result.scores.assign(panel=name));dynamic_boots.append(result.bootstrap.assign(panel=name));dynamic_factor_scores.append(result.factor_scores.assign(panel=name))
        status.append({'panel':name,'status':'completed',**result.metadata});print(name,result.metadata['first_oos'],result.metadata['last_oos']);display(result.scores.round(5))
        if traded and panel is not None and name in ('ETF_PROXY','FUNDED_RATES','JOINT_XA'):
            try:
                econ=economic_return_decomposition(pd.concat([ff_all,pricing.rolling_returns],axis=1),panel,as_of=AS_OF,window=DYNAMIC_WINDOW,min_train=DYNAMIC_MIN_TRAIN,step=DYNAMIC_REFIT)
                econ.to_csv(folder/'raw_traded_economic_decomposition.csv',index=False)
                # Report modeled contributions separately from retained intercept. Not causal shares.
                agg=econ.groupby('factor')[['intercept_monthly','driver_implied_monthly','total_expected_monthly','actual_monthly']].mean()*12
                agg.to_csv(folder/'raw_traded_economic_summary_ann.csv');display(agg.round(5))
            except ValueError as e:print('Economic decomposition not estimated:',e)
        del result
    except (ValueError,np.linalg.LinAlgError) as e:
        status.append({'panel':name,'status':'not estimated','detail':str(e)});print(name,'not estimated:',e)
for name,frames in [('dynamic_all_scores',dynamic_summaries),('dynamic_all_bootstrap',dynamic_boots),('dynamic_all_factor_scores',dynamic_factor_scores)]:
    if frames:pd.concat(frames,ignore_index=True).to_csv(OUT/(name+'.csv'),index=False)
pd.DataFrame(status).to_csv(OUT/'dynamic_experiment_status.csv',index=False)
print('New experiment completion:');display(pd.DataFrame(status)[['panel','status']])


## 17. Audit / next research decision

**Decision gates:** common residual risk, incremental OOS pricing after multiple-testing adjustment, pricing stability under rolling retraining, and pre-specified economic shock exposures/predictability must each be tested separately. Report uncertainty and any model selection as exploratory. This notebook does **not** claim published weak-factor robustness, identified inflation news when consensus is absent, or improved investable TAA value.

In [ ]:
print('ERP',__version__)
print('Core output:',OUT.resolve())
print('Rolling premia factors:',len(pricing.rolling_premia))
print('Pricing candidates x methods:',len(pricing.pricing_comparison))
print('FMB lambda rows:',len(pricing.fmb_lambdas))
print('GMM shrinkage status: diagnostics only; no standard J-test')
print('Policy shock mapping:', 'available' if shock_mapping is not None else 'not estimated')
print('CPI release shocks:', 'available' if inflation_shocks is not None else 'not estimated')

print("Dedicated rolling economic studies completed:",list(rolling_economic))
print("FRED proxies available:",macro is not None)

print("Dynamic experiments:",[(x["panel"],x["status"]) for x in status])
